# Fink/LSST — Download Object-Level Summary (`api/v1/objects`)

This notebook complements `01_fink_block_lightcurves.ipynb` by downloading data
from the **`api/v1/objects`** endpoint for all `diaObjectId` values that were
successfully retrieved in notebook 01 (those present in `lc_cache`, reflected in
`flatness_metrics.csv`).

## Why this endpoint?

The `/api/v1/objects` endpoint returns columns **aggregated at the object level**:
means, medians, and global statistics computed over all diaSources.
These columns are **distinct** from those returned by `/api/v1/sources` (per-detection)
and `/api/v1/fp` (per-epoch forced photometry).

The `fetch_objects()` function was already defined in notebook 01 but **never
called** in that workflow. This notebook uses it for all objects.

## Canonical source of objects

`flatness_metrics.csv` is the direct trace of `lc_cache` (section 7 of notebook 01):
it contains only the objects for which a light curve was successfully downloaded.
It is therefore the most reliable source for retrieving the exact list of
`diaObjectId` values to query.

```
flatness_metrics.csv  →  unique diaObjectId values (objects actually in lc_cache)
    ↓
POST https://api.lsst.fink-portal.org/api/v1/objects   →  summary per diaObjectId
    ↓
data_FINK_BLOCK_LC_01/objects_all.parquet   (+ .csv)
```


## 1. Imports & configuration

In [ ]:
import requests
import pandas as pd
import numpy as np
import os
import time
import warnings

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
# ── API base URL ──────────────────────────────────────────────────────────────
FINK_API = "https://api.lsst.fink-portal.org"

# ── Data directory (same as notebook 01) ─────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_01"
DIR_DATA = f"data_{NB_TAG}"

# Canonical source file: direct trace of lc_cache from notebook 01
FLATNESS_CSV = os.path.join(DIR_DATA, "flatness_metrics.csv")

# ── Delay between API requests (seconds) ─────────────────────────────────────
API_SLEEP = 0.25

# ── Maximum number of objects (None = all; set e.g. 10 for a quick test) ─────
MAX_OBJECTS = None

os.makedirs(DIR_DATA, exist_ok=True)
print(f"Data directory : {os.path.abspath(DIR_DATA)}")
print(f"Source CSV     : {os.path.abspath(FLATNESS_CSV)}")
assert os.path.exists(FLATNESS_CSV), f"{FLATNESS_CSV} not found — please run notebook 01 first."

## 2. API wrapper — `fetch_objects`

In [ ]:
def _post_json(url, payload, timeout=90):
    r = requests.post(url, json=payload, timeout=timeout)
    r.raise_for_status()
    return r.json()


def fetch_objects(diaObjectId, columns=None) -> pd.DataFrame:
    """
    Fetch object-level summary for one diaObjectId via /api/v1/objects.

    Returns a one-row DataFrame with all object-level columns returned by the API,
    or an empty DataFrame if the object is not found or the request fails.
    """
    payload = {"diaObjectId": str(diaObjectId), "output-format": "json"}
    if columns:
        payload["columns"] = columns
    try:
        raw = _post_json(f"{FINK_API}/api/v1/objects", payload)
        return pd.DataFrame(raw) if raw else pd.DataFrame()
    except requests.exceptions.HTTPError as e:
        print(f"  HTTP error for {diaObjectId}: {e}")
        return pd.DataFrame()
    except Exception as e:
        print(f"  Error for {diaObjectId}: {e}")
        return pd.DataFrame()


print("fetch_objects() defined.")

## 3. Load `diaObjectId` values from `flatness_metrics.csv`

`flatness_metrics.csv` is the direct trace of `lc_cache` in notebook 01:
it contains **only** the objects for which a light curve was successfully downloaded
(section 7 of notebook 01). It is exactly the same list of objects
as those saved in the parquet files `*_src.parquet` and `*_fp.parquet`.

In [ ]:
df_flat = pd.read_csv(FLATNESS_CSV)

# An object may appear multiple times (one row per band).
# Keep one row per diaObjectId, retaining the classification group.
df_oids = df_flat[["diaObjectId", "group"]].drop_duplicates(subset="diaObjectId").reset_index(drop=True)
df_oids["diaObjectId"] = df_oids["diaObjectId"].astype(str)

print(f"Source file        : {FLATNESS_CSV}")
print(f"Unique objects     : {len(df_oids)}  (from lc_cache in notebook 01)")
print(f"\nDistribution by group:")
print(df_oids["group"].value_counts().to_string())

if MAX_OBJECTS is not None:
    df_oids = df_oids.head(MAX_OBJECTS)
    print(f"\n[Limited to {MAX_OBJECTS} objects for testing]")

## 4. Probe the API: available columns in `/api/v1/objects`

Query a single object to display the full schema.

In [ ]:
sample_oid = df_oids["diaObjectId"].iloc[0]
print(f"Probing with diaObjectId = {sample_oid}")

df_sample = fetch_objects(sample_oid)

if df_sample.empty:
    print("WARNING: No data returned — check that /api/v1/objects is reachable.")
else:
    print(f"\nAvailable columns ({len(df_sample.columns)}):")
    for col in sorted(df_sample.columns):
        val = df_sample[col].iloc[0]
        print(f"  {col:55s}  {val}")

## 5. Download all objects

In [ ]:
objects_list = []
n_total = len(df_oids)
n_ok = 0
n_empty = 0

print(f"Downloading {n_total} objects via /api/v1/objects ...")
print("=" * 75)

for i, row in enumerate(df_oids.itertuples(index=False)):
    oid = str(row.diaObjectId)
    group = row.group

    df_obj = fetch_objects(oid)

    if df_obj.empty:
        n_empty += 1
        status = "EMPTY"
    else:
        # Prepend context columns inherited from notebook 01
        df_obj.insert(0, "diaObjectId", oid)
        df_obj.insert(1, "group", group)
        objects_list.append(df_obj)
        n_ok += 1
        status = f"OK  ncols={len(df_obj.columns)}"

    # Print progress every 10 iterations
    if (i + 1) % 10 == 0 or i == 0 or (i + 1) == n_total:
        print(f"  [{i + 1:4d}/{n_total}] {oid}  group={group:35s}  {status}")

    time.sleep(API_SLEEP)

print("\n" + "=" * 75)
print(f"OK={n_ok}   empty={n_empty}")

## 6. Assemble and inspect the global DataFrame

In [ ]:
if not objects_list:
    raise RuntimeError("No objects downloaded successfully — check the API.")

df_objects = pd.concat(objects_list, ignore_index=True)

# Drop any duplicate diaObjectId columns that may be returned by the API
dup_cols = [c for c in df_objects.columns if c != "diaObjectId" and c.lower() == "diaobjectid"]
if dup_cols:
    df_objects.drop(columns=dup_cols, inplace=True)

print(f"Final DataFrame : {len(df_objects)} rows × {len(df_objects.columns)} columns")
print(f"Unique objects  : {df_objects['diaObjectId'].nunique()}")
print("\nDtypes:")
print(df_objects.dtypes.to_string())

In [ ]:
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 220)
df_objects.head(5)

In [ ]:
df_objects.describe(include="all").T

## 7. Column inventory — fill rate

In [ ]:
n = len(df_objects)
fill_rate = df_objects.notna().sum().rename("n_non_null").to_frame()
fill_rate["fill_pct"] = (fill_rate["n_non_null"] / n * 100).round(1)
fill_rate["dtype"] = df_objects.dtypes
fill_rate = fill_rate.sort_values("fill_pct", ascending=False)

print(f"Column inventory ({len(fill_rate)}):\n")
print(fill_rate.to_string())

## 8. Distribution by group

In [ ]:
print("Objects per classification group (inherited from notebook 01):")
print(df_objects.groupby("group")["diaObjectId"].count().sort_values(ascending=False).to_string())

## 9. Save to disk

In [ ]:
# ── Parquet (primary format — preserves dtypes) ───────────────────────────────
path_parquet = os.path.join(DIR_DATA, "objects_all.parquet")
df_objects.to_parquet(path_parquet, index=False)
print(f"Saved: {path_parquet}  ({os.path.getsize(path_parquet) / 1e6:.2f} MB)")

# ── CSV (for quick inspection / broad compatibility) ──────────────────────────
path_csv = os.path.join(DIR_DATA, "objects_all.csv")
df_objects.to_csv(path_csv, index=False)
print(f"Saved: {path_csv}  ({os.path.getsize(path_csv) / 1e6:.2f} MB)")

print("\nDone.")

## 10. Read-back verification

In [ ]:
df_check = pd.read_parquet(path_parquet)
print(f"Read-back OK : {len(df_check)} rows × {len(df_check.columns)} columns")
print(f"Unique diaObjectId : {df_check['diaObjectId'].nunique()}")
print(f"Columns : {df_check.columns.tolist()}")
df_check.head(3)